# Phase 5: MITRE ATT&CK Integration Analysis
## UNSW-NB15 ML-SIEM — Threat Enrichment Layer

This notebook demonstrates how the **AttackEnricher** transforms raw ML detections
from Phase 4 into actionable threat intelligence by mapping them to MITRE ATT&CK
techniques, tactics, and descriptions.

### Enrichment Pipeline
```
EnsembleResult (Phase 4)
    ↓  AttackEnricher.enrich()
EnrichedIncident
    ├── verdict, confidence, severity   (from Phase 4)
    ├── primary_technique               (ATT&CK technique object)
    ├── tactic / tactic_id             (ATT&CK tactic)
    ├── mapping_rationale              (why this mapping was chosen)
    └── secondary_techniques           (additional relevant techniques)
```

### Mapping Coverage
| Category | Primary Technique | Tactic | Confidence |
|---|---|---|---|
| Reconnaissance | T1595 Active Scanning | Reconnaissance | HIGH |
| DoS | T1498 Network Denial of Service | Impact | HIGH |
| Exploits | T1190 Exploit Public-Facing Application | Initial Access | MEDIUM |
| Backdoors | T1505 Server Software Component | Persistence | MEDIUM |
| Shellcode | T1059 Command and Scripting Interpreter | Execution | MEDIUM |
| Worms | T1210 Exploitation of Remote Services | Lateral Movement | MEDIUM |
| Fuzzers | T1190 Exploit Public-Facing Application | Initial Access | MEDIUM |
| Analysis | T1040 Network Sniffing | Credential Access | LOW |
| Generic | T1190 Exploit Public-Facing Application | Initial Access | LOW |
| Anomaly | T1190 (fallback) | Initial Access | LOW |
| unknown | — (normal traffic) | N/A | HIGH |


In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.attack_mapping import (
    AttackEnricher, get_attack_data, validate_mapping_config,
    get_coverage_summary, stix_info
)
from src.ensemble import EnsembleDetector

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print("✅ Imports OK — PROJECT_ROOT:", PROJECT_ROOT)


## 1. Load Validation Report

In [ ]:
VAL_PATH = PROJECT_ROOT / 'reports' / 'attack_mapping_validation.json'
assert VAL_PATH.exists(), f"Run run_attack_mapping.py first: {VAL_PATH}"

with open(VAL_PATH) as f:
    report = json.load(f)

info    = report['stix_info']
val     = report['validation']
stats   = report['enrichment_statistics']
cov     = val['coverage']

print(f"ATT&CK techniques loaded : {info['n_techniques']}")
print(f"ATT&CK tactics loaded    : {info['n_tactics']}")
print(f"Categories mapped        : {cov['mapped_categories']} / {cov['total_categories']}")
print(f"Coverage                 : {cov['mapping_coverage_pct']}%")
print(f"Valid technique IDs      : {cov['valid_technique_count']}")
print(f"Validation               : {'PASSED ✅' if cov['validation_passed'] else 'FAILED ❌'}")
print()
print("Tactic distribution (sample):")
for tac, cnt in sorted(stats['tactic_distribution'].items(), key=lambda x: -x[1]):
    print(f"  {tac:30s}: {cnt:,}")


## 2. Load Sample Enriched Incidents

In [ ]:
SAMPLES_PATH = PROJECT_ROOT / 'reports' / 'incidents' / 'sample_enriched.json'
with open(SAMPLES_PATH) as f:
    samples = json.load(f)

print(f"Sample incidents: {len(samples)}")
print()
for s in samples:
    cat    = s.get('_ground_truth_category', 'N/A')
    verd   = s.get('verdict', '?')
    tech   = (s.get('primary_technique') or {}).get('technique_id', 'N/A')
    tname  = (s.get('primary_technique') or {}).get('name', 'N/A')
    tactic = s.get('tactic', 'N/A')
    sev    = s.get('severity', 'N/A')
    conf   = s.get('mapping_confidence', 'N/A')
    print(f"  [{cat:17s}] verdict={verd:6s} tech={tech:12s} ({tname[:30]:<30s}) tactic={tactic:22s} sev={sev:8s} map_conf={conf}")


## 3. Detailed Enriched Incident Examples

In [ ]:
def print_incident(sample: dict, title: str = "") -> None:
    print(f"\n{'='*60}")
    print(f"  {title or sample.get('_ground_truth_category', 'Incident')}")
    print(f"{'='*60}")
    print(f"  Verdict         : {sample.get('verdict')}")
    print(f"  Confidence      : {sample.get('confidence', 0):.4f}")
    print(f"  Severity        : {sample.get('severity')}")
    print(f"  Agreement Score : {sample.get('agreement_score', 0):.4f}")
    print(f"  Attack Category : {sample.get('attack_category')}")
    print()
    print(f"  ── ATT&CK Enrichment ──────────────────────────────────")
    tech = sample.get('primary_technique') or {}
    print(f"  Technique ID    : {tech.get('technique_id', 'N/A')}")
    print(f"  Technique Name  : {tech.get('name', 'N/A')}")
    print(f"  Tactic          : {sample.get('tactic')} ({sample.get('tactic_id')})")
    print(f"  Platforms       : {', '.join(tech.get('platforms', []))}")
    print(f"  ATT&CK URL      : {tech.get('url', 'N/A')}")
    print(f"  Map Confidence  : {sample.get('mapping_confidence')}")
    print(f"  Is Mapped       : {sample.get('is_mapped')}")
    print(f"\n  Rationale: {sample.get('mapping_rationale', '')[:200]}")
    secs = sample.get('secondary_techniques', [])
    if secs:
        print(f"\n  Secondary Techniques:")
        for st in secs:
            print(f"    - {st.get('technique_id','?')} | {st.get('name','?')}")

# Print a few key scenarios
for cat in ['unknown', 'Reconnaissance', 'Exploits', 'DoS', 'Backdoors']:
    match = next((s for s in samples if s.get('_ground_truth_category') == cat), None)
    if match:
        print_incident(match, f"{cat} Sample")


## 4. Mapping Configuration Visualisation

In [ ]:
# Category → Technique → Tactic mapping table
mapping_data = []
for s in samples:
    cat = s.get('_ground_truth_category', '?')
    tech = (s.get('primary_technique') or {})
    mapping_data.append({
        'Category': cat,
        'Technique ID': tech.get('technique_id', 'N/A'),
        'Technique Name': tech.get('name', 'N/A')[:35],
        'Tactic': s.get('tactic', 'N/A'),
        'Mapping Confidence': s.get('mapping_confidence', '?'),
        'Is Mapped': '✅' if s.get('is_mapped') else '—',
    })

df_map = pd.DataFrame(mapping_data)
print(df_map.to_string(index=False))


In [ ]:
# Visualise mapping confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Bar chart of confidence levels
conf_colors = {'HIGH': '#2ecc71', 'MEDIUM': '#f39c12', 'LOW': '#e74c3c'}
conf_data   = {k: int(v) for k, v in val.get('coverage', {}).get('confidence_breakdown', {}).items()}
bars = axes[0].bar(conf_data.keys(), conf_data.values(),
                   color=[conf_colors.get(k,'grey') for k in conf_data], edgecolor='white')
for bar, v in zip(bars, conf_data.values()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set(title='Category Mapping Confidence Distribution',
            xlabel='Mapping Confidence', ylabel='Number of Categories')
axes[0].set_ylim(0, max(conf_data.values())*1.3)

# Right: Mapped vs Unmapped
mapped_n   = cov['mapped_categories']
unmapped_n = cov['unmapped_categories']
axes[1].pie([mapped_n, unmapped_n],
            labels=['Mapped', 'Unmapped (Normal)'],
            colors=['#3498db','#95a5a6'],
            autopct='%1.0f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('ATT&CK Mapping Coverage')

plt.suptitle('Fig 1: ATT&CK Mapping Coverage', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_mapping_coverage.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Live Enrichment on Test Sample

In [ ]:
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
feat_names = (PROCESSED/'feature_list.txt').read_text().strip().splitlines()
df_test    = pd.read_parquet(PROCESSED/'test.parquet',
                              columns=feat_names+['label','attack_cat'])
df_s       = df_test.sample(n=5000, random_state=42).reset_index(drop=True)
X_s        = df_s[feat_names].values.astype('float32')
print(f"Sample: {len(df_s):,} rows")


In [ ]:
ensemble = EnsembleDetector.from_disk(
    PROJECT_ROOT/'data/models/rf_detector.joblib',
    PROJECT_ROOT/'data/models/xgb_detector.joblib',
    PROJECT_ROOT/'models/isolation_forest.joblib',
    PROJECT_ROOT/'models/autoencoder.pt',
    batch_size=2048
)
enricher  = AttackEnricher(stix_path=str(PROJECT_ROOT/'data/attack/enterprise-attack.json'))
results   = ensemble.predict(X_s)
incidents = enricher.enrich_batch(results)
print(f"Enriched {len(incidents):,} incidents")


In [ ]:
# Build analysis DataFrame
rows = []
for i, inc in enumerate(incidents):
    tech = inc.primary_technique
    rows.append({
        'true_cat': df_s['attack_cat'].iloc[i],
        'true_label': int(df_s['label'].iloc[i]),
        'verdict': inc.verdict,
        'confidence': inc.confidence,
        'severity': inc.severity,
        'attack_cat': inc.attack_category,
        'tactic': inc.tactic,
        'tactic_id': inc.tactic_id,
        'technique_id': tech.technique_id if tech else 'N/A',
        'technique_name': tech.name if tech else 'N/A',
        'mapping_confidence': inc.mapping_confidence,
        'is_mapped': inc.is_mapped,
    })
df = pd.DataFrame(rows)
print(df['tactic'].value_counts().to_string())


## 6. Tactic Distribution

In [ ]:
attack_df = df[df['verdict']=='ATTACK']
tactic_counts = attack_df['tactic'].value_counts()

tactic_palette = {
    'Initial Access':'#e74c3c','Reconnaissance':'#9b59b6','Impact':'#e67e22',
    'Persistence':'#f39c12','Lateral Movement':'#3498db','Execution':'#1abc9c',
    'Credential Access':'#2c3e50','N/A':'#95a5a6','Unknown':'#bdc3c7',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = [tactic_palette.get(t,'#7f8c8d') for t in tactic_counts.index]
bars = axes[0].barh(tactic_counts.index, tactic_counts.values, color=colors, edgecolor='white')
for bar, v in zip(bars, tactic_counts.values):
    axes[0].text(bar.get_width()+2, bar.get_y()+bar.get_height()/2,
                 f'{v:,}', va='center', fontsize=9)
axes[0].set(title='ATT&CK Tactic Distribution (ATTACK alerts)',
            xlabel='Count', ylabel='Tactic')

axes[1].pie(tactic_counts.values,
            labels=[f'{t}
({v:,})' for t, v in tactic_counts.items()],
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 8})
axes[1].set_title('Tactic Share')

plt.suptitle('Fig 2: ATT&CK Tactic Distribution', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_tactic_dist.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Technique Frequency

In [ ]:
tech_counts = attack_df['technique_id'].value_counts().head(15)
tech_names  = attack_df.drop_duplicates('technique_id').set_index('technique_id')['technique_name']

fig, ax = plt.subplots(figsize=(11, 5))
colors  = plt.cm.viridis(np.linspace(0.2, 0.9, len(tech_counts)))
bars    = ax.bar(range(len(tech_counts)), tech_counts.values, color=colors, edgecolor='white')

labels = [f"{tid}\n{tech_names.get(tid,'')[:20]}" for tid in tech_counts.index]
ax.set_xticks(range(len(tech_counts)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
for bar, v in zip(bars, tech_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'{v:,}', ha='center', fontsize=8)
ax.set(title='Top ATT&CK Techniques by Frequency', xlabel='Technique', ylabel='Count')

plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_technique_freq.png', dpi=120, bbox_inches='tight')
plt.show()


## 8. Severity Distribution After Enrichment

In [ ]:
sev_order  = ['CRITICAL','HIGH','MEDIUM','LOW','N/A']
sev_colors = ['#c0392b','#e67e22','#f39c12','#3498db','#95a5a6']
sev_counts = df['severity'].value_counts().reindex(sev_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(sev_counts.index, sev_counts.values, color=sev_colors, edgecolor='white')
for bar, v in zip(bars, sev_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                 f'{v:,}', ha='center', fontsize=9)
axes[0].set(title='Severity Distribution (All)', xlabel='Severity', ylabel='Count')

# Severity by tactic (ATTACK only)
sev_tac = attack_df.groupby(['tactic','severity']).size().unstack(fill_value=0)
sev_tac = sev_tac.reindex(columns=[c for c in sev_order if c in sev_tac.columns])
sev_tac.plot(kind='bar', ax=axes[1], color=sev_colors[:len(sev_tac.columns)],
             edgecolor='white')
axes[1].set(title='Severity by Tactic', xlabel='Tactic', ylabel='Count')
axes[1].tick_params(axis='x', rotation=25)
axes[1].legend(title='Severity', fontsize=8)

plt.suptitle('Fig 3: Severity Distribution After ATT&CK Enrichment', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_severity_enriched.png', dpi=120, bbox_inches='tight')
plt.show()


## 9. Category → Tactic Flow Diagram

In [ ]:
# Sankey-style: attack_cat → tactic heatmap
cat_tac = attack_df.groupby(['attack_cat','tactic']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(cat_tac, annot=True, fmt='d', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Count'})
ax.set(title='Attack Category → ATT&CK Tactic Mapping',
       xlabel='ATT&CK Tactic', ylabel='UNSW-NB15 Attack Category')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(PROJECT_ROOT/'reports'/'fig_cat_tactic_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Mapping Rationale Summary (Viva Defence)

### Why These Mappings?

| Category | ATT&CK Technique | Tactic | Confidence | Key Rationale |
|---|---|---|---|---|
| Reconnaissance | T1595 Active Scanning | Reconnaissance | HIGH | Port sweeps = ATT&CK's definition of active scanning; external pre-attack phase |
| DoS | T1498 Network DoS | Impact | HIGH | Volumetric flooding = availability impact; resource exhaustion is the adversary goal |
| Exploits | T1190 Exploit Public-Facing App | Initial Access | MEDIUM | Network exploit against public services = initial foothold attempt |
| Backdoors | T1505 Server Software Component | Persistence | MEDIUM | Server-side implants (web shells, SSH tunnels) maintain persistent access |
| Shellcode | T1059 Command Interpreter | Execution | MEDIUM | Shellcode spawns shells; Execution is the resulting adversary capability |
| Worms | T1210 Exploit Remote Services | Lateral Movement | MEDIUM | Self-propagation via vulnerability = host-to-host lateral movement |
| Fuzzers | T1190 Exploit Public-Facing App | Initial Access | MEDIUM | Intent is vulnerability discovery leading to exploitation |
| Analysis | T1040 Network Sniffing | Credential Access | LOW | Passive capture tools; "Analysis" label is broad and ambiguous |
| Generic | T1190 Exploit Public-Facing App | Initial Access | LOW | Catch-all category; best representative is generic exploit activity |

### Mapping Limitations (Important for Viva)
1. **Dataset-level labels ≠ individual technique instances** — UNSW-NB15 labels a
   *traffic stream* not a specific exploit. A real ATT&CK mapping would require
   packet-level inspection (CVE number, payload analysis).

2. **"Generic" is inherently heterogeneous** — it contains multiple technique types;
   our mapping picks the plurality representative but cannot be precise.

3. **ATT&CK version pinning** — mappings are valid for ATT&CK Enterprise v14.
   Technique IDs may change across versions (e.g. T1086 → T1059.001 in v8).

4. **Unsupervised-only detections** — when supervised models vote Normal but anomaly
   detectors fire (category='Anomaly'), we cannot categorise the technique —
   this is the zero-day scenario and is correctly flagged as LOW confidence.


In [ ]:
# Final summary
print("=" * 60)
print(" PHASE 5 ENRICHMENT SUMMARY")
print("=" * 60)
print(f"  ATT&CK techniques (STIX): {info['n_techniques']}")
print(f"  Categories mapped       : {cov['mapped_categories']}/{cov['total_categories']}")
print(f"  Valid technique IDs     : {cov['valid_technique_count']}")
print(f"  Validation              : {'PASSED' if cov['validation_passed'] else 'FAILED'}")
print()
print("  Sample tactic distribution:")
for t, v in sorted(stats['tactic_distribution'].items(), key=lambda x: -x[1]):
    print(f"    {t:30s}: {v:,}")
print()
print("  Mapping confidence breakdown:")
for k, v in cov['confidence_breakdown'].items():
    print(f"    {k:10s}: {v}")
print()
print("  Saved figures: reports/fig_mapping_*.png, fig_tactic_*.png, etc.")
